<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/50_ode/55_Spread_of_disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Simulating Spread of Disease<br>질병 전파 시뮬레이션


Ref : David Smith and Lang Moore, "The SIR Model for Spread of Disease - The Differential Equation Model," Convergence (December 2004),  https://www.maa.org/press/periodicals/loci/joma/the-sir-model-for-spread-of-disease-the-differential-equation-model



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.linalg as nl
import scipy.integrate as si



* We would consider three types of people.<br>3종류의 사람들로 구분할 것이다.



| variable<br>변수 | description<br>설명 |
|:-----:|:-----:|
| $s$ | the susceptible fraction of the population<br>감염될 수 있는 인구 비중 |
| $i$ | the infected fraction of the population<br>감염된 인구 비중 |
| $r$ | the recovered fraction of the population<br>회복된 인구 비중 |



$$
\begin{align}
\frac{ds}{dt}&=-b\cdot s(t)i(t) \\
\frac{di}{dt}&=b\cdot s(t)i(t)-k\cdot i(t) \\
\frac{dr}{dt}&=k\cdot i(t) \\
\end{align}
$$



| variable<br>변수 |   value<br>값    |   description<br>설명    |
|:---------------:|:---------------:|:---------------:|
| $b$ | 1/2 | transmission rate<br>전염률 |
| $k$ | 1/2 | recovery rate<br>회복률 |



* If these parameters change, what happens? Maybe simulations can show us.<br>이러한 매개변수들 값이 달라지면 어떻게 될 것인가? 시뮬레이션으로 확인해 보는 건 어떨까?



In [ ]:
b = 0.5
k = 1.0/3



* The following video illustrates how the susceptible, infected, and recovered populations change over time under different parameter values (23:11)<br>아래 비디오에서 해당 SIR 모델이 매개변수에 따라 어떻게 다르게 거동하는지 관찰해 볼 수 있다. (23:11)

[![video here](https://i.ytimg.com/vi/gxAaO2rsdIs/hqdefault.jpg)](https://youtu.be/gxAaO2rsdIs)



### Numerical Solution using `solve_ivp()`<br>`solve_ivp()` 수치해



* Now let's implement the SIR model above and simulate.<br>이제 위 SIR 모델을 구현해 시험해 봅시다.
* Slope function<br>기울기 함수



In [ ]:
def dy_dt(t, y):
  s = y[0]
  i = y[1]
  r = y[2]
  return np.array((
      -b * s * i,
      b * s * i - k * i,
      k * i,
  ))



* Simulation<br>시뮬레이션



In [ ]:
delta_t = 1 # day
t_start = 0
t_end = 150 # day

# Why the initial condition as follows?
# 왜 이러한 초기값을 사용하는가?
y0 = [1, 1.27e-6, 0]

t_eval = np.arange(t_start, t_end, delta_t)

sol = si.solve_ivp(dy_dt, (t_start, t_end), y0=y0, t_eval=t_eval)



In [ ]:
sol



* Let's plot.<br>그림으로 표시해 보자.



In [ ]:
ylabels = ['s', 'i', 'r']

for j, y in enumerate(sol.y):

  ax = plt.subplot(len(sol.y), 1, j+1)
  ax.plot(sol.t, y)
  ax.set_ylabel(f'${ylabels[j]}(t)$')
  ax.grid(True)

ax.set_xlabel('$t$ (days)');



* What about stacking all three plots?<br>세 그래프를 수직으로 쌓아 보면 어떨까?



In [ ]:
ylabels = ['s', 'i', 'r']

plt.clf()
plt.stackplot(sol.t, sol.y)
plt.xlabel('$t$ (days)');



## The big number: $R_0 = b / k$ <br>핵심 숫자: $R_0 = b / k$


If one infected person stays sick for about $1/k$ days, and infects $b$ new people per day (when almost everyone is susceptible), then **one infected person triggers about $b/k$ more infections in total**. We call this number $R_0$ (R-naught).<br>한 명의 감염자가 약 $1/k$ 일 동안 아프다 회복하며, 그 사이 하루에 $b$ 명에게 옮긴다고 가정하면 (거의 모든 사람이 감수성자일 때), **한 명의 감염자는 평균 $b/k$ 명에게 병을 옮긴다**. 이 숫자를 $R_0$ (R-naught, 기초감염재생산수) 이라고 부른다.

* $R_0 < 1$ → each sick person infects fewer than one new person → outbreak fizzles out.<br>$R_0 < 1$ → 한 환자가 한 명 미만에게 옮김 → 유행이 곧 사그라든다.
* $R_0 > 1$ → infections grow exponentially at first → real outbreak.<br>$R_0 > 1$ → 처음에 감염자 수가 지수적으로 증가 → 실제 유행.
* $R_0 = 1$ → the knife edge.<br>$R_0 = 1$ → 경계선.


Let's see all three regimes side by side.<br>세 가지 경우를 나란히 비교해 보자.


In [ ]:
scenarios = [
    ('R_0 = 0.6 (outbreak fizzles)',        0.2, 1/3),
    ('R_0 = 1.5 (outbreak grows then peaks)', 0.5, 1/3),
    ('R_0 = 3.0 (fast outbreak)',           1.0, 1/3),
]

fig, axes = plt.subplots(len(scenarios), 1, figsize=(9, 9), sharex=True)

for ax, (label, b_val, k_val) in zip(axes, scenarios):
    def slope(t, y, b=b_val, k=k_val):
        s, i, r = y
        return np.array((-b*s*i, b*s*i - k*i, k*i))

    sol = si.solve_ivp(slope, (0, 150), [1.0, 1.27e-6, 0.0],
                       t_eval=np.arange(0, 150, 1))
    ax.plot(sol.t, sol.y[0], label='s (susceptible)')
    ax.plot(sol.t, sol.y[1], label='i (infected)')
    ax.plot(sol.t, sol.y[2], label='r (recovered)')
    ax.set_title(label)
    ax.set_ylabel('fraction')
    ax.grid(True)
    ax.legend(loc='right')

axes[-1].set_xlabel('$t$ (days)')
plt.tight_layout()
plt.show()


**Notice**: the curves for $R_0 = 0.6$ stay nearly flat — the infected fraction barely rises. For $R_0 = 1.5$ and $R_0 = 3.0$ the infected peak is real, and the steeper $R_0$ peaks earlier and higher.<br>**관찰**: $R_0 = 0.6$ 인 경우 곡선이 거의 평평하다 — 감염자 비중이 거의 늘지 않는다. $R_0 = 1.5$ 와 $R_0 = 3.0$ 에서는 감염자 비중에 뚜렷한 정점이 나타나고, $R_0$ 가 클수록 정점이 더 일찍 더 높이 형성된다.


### Herd immunity threshold <br>집단 면역 임계점


Once enough people have recovered (and become immune), each new infection meets so few susceptible people that the outbreak can't grow. The fraction of people who need to be immune for the outbreak to stop growing is $1 - 1/R_0$.<br>충분히 많은 사람이 회복되어 면역을 얻으면, 새 감염자가 만나는 감수성자가 너무 적어 유행이 더 자랄 수 없다. 이때 필요한 면역자 비중은 $1 - 1/R_0$ 이다.

* $R_0 = 2$ → need 50% immune to stop growth.<br>$R_0 = 2$ → 50% 가 면역이어야 멈춤.
* $R_0 = 3$ → need 67% immune.<br>$R_0 = 3$ → 67% 가 면역이어야 멈춤.
* $R_0 = 10$ (measles 홍역) → need 90% immune.<br>$R_0 = 10$ (홍역) → 90% 가 면역이어야 멈춤.


## Play with the parameters: $b$ and $k$ <br>매개변수를 직접 조절해 보자


In [ ]:
from ipywidgets import interact, FloatSlider

def plot_sir(b=0.5, k=1.0/3):
    def slope(t, y):
        s, i, r = y
        return np.array((-b*s*i, b*s*i - k*i, k*i))

    sol = si.solve_ivp(slope, (0, 150), [1.0, 1.27e-6, 0.0],
                       t_eval=np.arange(0, 150, 1))
    R0 = b / k

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(sol.t, sol.y[0], label='s (susceptible)')
    ax.plot(sol.t, sol.y[1], label='i (infected)')
    ax.plot(sol.t, sol.y[2], label='r (recovered)')
    ax.set_xlabel('$t$ (days)')
    ax.set_ylabel('fraction')
    ax.set_title(f'b={b:.2f}, k={k:.2f}  ->  R_0 = b/k = {R0:.2f}')
    ax.grid(True)
    ax.legend()
    plt.show()

interact(
    plot_sir,
    b=FloatSlider(min=0.05, max=2.0,  step=0.05, value=0.5),
    k=FloatSlider(min=0.05, max=1.0,  step=0.05, value=1.0/3),
);


## Exercise: add a vaccinated group (SVIR model) <br>연습 문제 : 백신접종군을 추가한 SVIR 모델


In the original SIR model, only sick people can become immune (after they recover). What if some fraction of the susceptible people get **vaccinated**, gaining immunity *without* getting sick first?<br>원래 SIR 모델에서는 환자가 회복되어야만 면역을 얻는다. 만약 감수성자 중 일부가 **백신을 맞고** 아프지 않은 채로 면역을 얻는다면 어떻게 될까?

Add a fourth compartment $v$ (vaccinated). Let $\nu$ (vaccination rate) be the daily fraction of susceptibles vaccinated:<br>네 번째 구획 $v$ (vaccinated) 를 추가한다. $\nu$ (백신접종률) 는 하루에 감수성자 중 백신을 맞는 비중:

$$
\begin{align}
\frac{ds}{dt} &= -b s i - \nu s \\
\frac{di}{dt} &= b s i - k i \\
\frac{dr}{dt} &= k i \\
\frac{dv}{dt} &= \nu s
\end{align}
$$

Implement and simulate. Then answer: with $b=0.5$, $k=1/3$ ($R_0 = 1.5$), what value of $\nu$ keeps the infected peak below 5% of the population?<br>구현하여 시뮬레이션해 보자. 그리고 답해 보자: $b=0.5$, $k=1/3$ ($R_0 = 1.5$) 일 때, 감염자 정점이 인구의 5% 미만이 되려면 $\nu$ 는 얼마여야 하는가?


In [ ]:
# Your code here
# 여기에 코드를 작성하세요
#
# Hint: copy the slope function above and add the -nu*s and +nu*s terms.
# Initial condition becomes [s0, i0, r0, v0] = [1.0, 1.27e-6, 0.0, 0.0].
# Try nu = 0.01, 0.02, 0.05 and watch the infected peak shrink.

nu = 0.01

def slope_svir(t, y):
    s, i, r, v = y
    return np.array((
        0,  # ds/dt = ?
        0,  # di/dt = ?
        0,  # dr/dt = ?
        0,  # dv/dt = ?
    ))

# Then call si.solve_ivp(...) and plot.

assert nu > 0


## Wishing you and everyone good health<br>여러분 모두의 건강을 빕니다.
